# 04 Baselines: simple references (E09)

**Purpose.** Score the validation set with simple, untrained rankings, each sorting the candidates by one signal. They are the reference points every trained model is compared with, and they show how strong popularity, relevance and personal history are on their own. Team 3's model and its correction ladder follow in a later part of this notebook.

**Which references** (fixed in advance from training-row evidence, not chosen on validation):

| Reference | Feature | Why |
|---|---|---|
| Random order | constant score | floor (E02) |
| BM25 | `bm25` | standard lexical relevance |
| Query-title Jaccard | `title_jaccard` | best lexical signal on training rows (E07) |
| Training exposures | `pop_exposures` | strongest non-history popularity on training rows (E07, E08) |
| History popularity | `pop_history` | strongest single signal on training rows (E08) |
| Engagement rate | `pop_engaged_rate` | supervised popularity |
| Same product, all history | `pooled_product` | best personal signal (E03) |
| Same shop, other products | `pooled_shop_other` | personal evidence that transfers to new products (E05b) |
| Same product, recent clicks | `click_0_7_product` | strongest window block (E03) |
| Same product, query-related history | `qrel_product` | query relevance of history (Q1) |

**Rules.**
- Validation searches with at least one engaged candidate (19,772), complete lists, exact expectation over random tie order (`scripts/evaluation.py`). Many candidates tie (e.g. zero popularity); ties are never broken by file order.
- Missing values (`NaN`, unknown metadata) are scored as 0: unknown counts as "no evidence", neither better nor worse than no match.
- Nothing is tuned here. Each result goes into `docs/experiment_results.csv` as experiment E09.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/evaluation.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
PROCESSED = REPO_ROOT / "data/processed"

sys.path.insert(0, str(REPO_ROOT))
from scripts.evaluation import RESULTS, evaluate, log_result, paired_difference, summarise

pd.options.display.float_format = "{:,.4f}".format
FEATURES = "v2"
meta = json.loads((PROCESSED / f"features_{FEATURES}_meta.json").read_text())
searches = pd.read_parquet(PROCESSED / "search_table.parquet")
assignment = pd.read_csv(PROCESSED / "split_v1.csv", index_col="session_id")
validation_ids = assignment.index[assignment.split == "validation"]
validation_sizes = searches.loc[validation_ids, "n_candidates"]

## 1. Load the validation features

In [ ]:
REFERENCES = {
    "random order": None,
    "BM25": "bm25",
    "query-title Jaccard": "title_jaccard",
    "training exposures": "pop_exposures",
    "history popularity": "pop_history",
    "engagement rate": "pop_engaged_rate",
    "same product, all history": "pooled_product",
    "same shop, other products": "pooled_shop_other",
    "same product, recent clicks": "click_0_7_product",
    "same product, query-related history": "qrel_product",
}
columns = ["session_id", "label"] + [f for f in REFERENCES.values() if f]
validation = pd.read_parquet(PROCESSED / f"features_{FEATURES}_validation.parquet", columns=columns)
print(f"validation: {validation.session_id.nunique():,} searches, {len(validation):,} candidates")
validation[[f for f in REFERENCES.values() if f]].isna().mean().rename("missing share").to_frame()

## 2. Evaluate every reference

In [ ]:
per_search = {}
for name, feature in REFERENCES.items():
    validation["score"] = 0.0 if feature is None else validation[feature].fillna(0).astype(float)
    per_search[name] = evaluate(validation, "score", expected_sizes=validation_sizes)

results = pd.concat({name: summarise(per).iloc[0] for name, per in per_search.items()}, axis=1).T
results["searches"] = results.searches.astype(int)
results = results.sort_values("ndcg_at_10", ascending=False)
results

Paired differences on the same searches (mean and 95% bootstrap interval of the per-search difference): against random order, and against history popularity, the strongest non-personal reference on training rows.

In [ ]:
rows = {}
for name in results.index:
    vs_random = paired_difference(per_search[name], per_search["random order"])
    vs_popularity = paired_difference(per_search[name], per_search["history popularity"])
    rows[name] = {"vs_random": vs_random[0], "vs_random_low": vs_random[1], "vs_random_high": vs_random[2],
                  "vs_history_pop": vs_popularity[0], "vs_history_pop_low": vs_popularity[1],
                  "vs_history_pop_high": vs_popularity[2]}
differences = pd.DataFrame(rows).T
differences

## 3. Log to the results registry

One row per reference (experiment E09). Re-running the notebook does not duplicate rows.

In [ ]:
logged = pd.read_csv(RESULTS) if RESULTS.exists() else pd.DataFrame(columns=["experiment", "model"])
for name, feature in REFERENCES.items():
    if name == "random order" or ((logged.experiment == "E09") & (logged.model == name)).any():
        continue  # random order is already logged as E02
    log_result("E09", name, "validation", per_search[name],
               notes=f"untrained: sort by {feature} (features {FEATURES}); NaN as 0")
pd.read_csv(RESULTS)

## 4. By segment

The random-order floor differs between segments (E02), so each segment is read as the gain over random **within** that segment.

In [ ]:
info = searches.loc[per_search["random order"].index]
segments = {
    "activity in last 7 days": np.where(info.days_since_last_action < 7, "yes", "no"),
    "list size": pd.cut(info.n_candidates, [0, 30, 90, 150, 200], labels=["1-30", "31-90", "91-150", "151-200"]),
    "history length": pd.cut(info.history_len, [0, 30, 100, 300, np.inf], labels=["1-30", "31-100", "101-300", "300+"]),
}
shown = ["history popularity", "training exposures", "same product, all history", "same product, recent clicks",
         "query-title Jaccard"]
tables = {}
for segment, labels in segments.items():
    by = pd.Series(labels, index=info.index)
    random_mean = per_search["random order"].ndcg.groupby(by, observed=True).mean()
    table = pd.DataFrame({name: per_search[name].ndcg.groupby(by, observed=True).mean() - random_mean for name in shown})
    table.insert(0, "searches", by.value_counts())
    table.insert(1, "random", random_mean)
    tables[segment] = table
pd.concat(tables)

## Summary

Filled in after running; see `docs/EXPERIMENT_LOG.md` (E09).